# Notebook 2: Conversational Chatbot with BlenderBot

In this notebook we build a multi-turn conversational chatbot using Facebook's **BlenderBot** model via the Hugging Face `transformers` library.

## Learning Objectives
- Understand how encoder-decoder dialogue models work
- Manage conversation history / context windows
- Use `BlenderbotTokenizer` and `BlenderbotForConditionalGeneration`
- Build an interactive command-line chat loop
- Discuss extension ideas (domain adaptation, retrieval-augmented chat)

## Background
BlenderBot (Roller et al., 2020) is trained on large-scale open-domain conversation data with a focus on **blending** multiple conversational skills: engaging personality, knowledge, and empathy. The 400M-parameter variant (`facebook/blenderbot-400M-distill`) is small enough for CPU inference.

## 1. Install & Import Dependencies

In [ ]:
# !pip install transformers torch

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import torch
from transformers import BlenderbotTokenizer, BlenderbotForConditionalGeneration

MODEL_NAME = "facebook/blenderbot-400M-distill"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## 2. Load the Model and Tokenizer

In [ ]:
print(f"Loading model '{MODEL_NAME}'...")
tokenizer = BlenderbotTokenizer.from_pretrained(MODEL_NAME)
model = BlenderbotForConditionalGeneration.from_pretrained(MODEL_NAME)
model = model.to(device)
model.eval()
print(f"Model loaded. Parameters: {model.num_parameters():,}")

## 3. Single-Turn Inference

Before building the full chatbot, let's do a single round-trip to make sure everything works.

In [ ]:
def get_reply(user_message: str, conversation_history: list[str] | None = None) -> str:
    """
    Generate a chatbot reply.

    Parameters
    ----------
    user_message : str
        The latest message from the user.
    conversation_history : list[str], optional
        Previous turns formatted as [user1, bot1, user2, bot2, ...].
        BlenderBot expects history tokens separated by '  ' (two spaces).

    Returns
    -------
    str
        The model's response.
    """
    if conversation_history:
        # Join history with the special separator that BlenderBot uses
        history_str = "  ".join(conversation_history)
        full_input = history_str + "  " + user_message
    else:
        full_input = user_message

    inputs = tokenizer(
        full_input,
        return_tensors="pt",
        truncation=True,
        max_length=128,
    ).to(device)

    with torch.no_grad():
        reply_ids = model.generate(
            **inputs,
            max_new_tokens=60,
            num_beams=3,
            early_stopping=True,
            no_repeat_ngram_size=3,
        )

    return tokenizer.decode(reply_ids[0], skip_special_tokens=True).strip()


# Quick test
reply = get_reply("Hello! What is your favourite hobby?")
print(f"User : Hello! What is your favourite hobby?")
print(f"Bot  : {reply}")

## 4. Multi-Turn Conversation

A proper chatbot maintains a conversation **history** so responses are contextually aware.

In [ ]:
class Chatbot:
    """A simple stateful BlenderBot chatbot."""

    def __init__(self, max_history_turns: int = 4):
        self.history: list[str] = []  # alternating [user, bot, user, bot, ...]
        self.max_history_turns = max_history_turns  # number of complete (user+bot) pairs to keep

    def chat(self, user_message: str) -> str:
        """Send a message and receive a response."""
        reply = get_reply(user_message, self.history)
        # Update history
        self.history.append(user_message)
        self.history.append(reply)
        # Trim to keep only the most recent turns
        max_tokens = self.max_history_turns * 2  # each turn = user + bot
        if len(self.history) > max_tokens:
            self.history = self.history[-max_tokens:]
        return reply

    def reset(self):
        """Start a fresh conversation."""
        self.history = []
        print("Conversation history cleared.")


# Simulate a multi-turn conversation
bot = Chatbot()

turns = [
    "Hi! My name is Alex. What is your name?",
    "That's cool! Do you like science fiction?",
    "Me too! What is your favourite sci-fi movie?",
    "Interesting choice. Have you read any science fiction books lately?",
]

for user_msg in turns:
    bot_reply = bot.chat(user_msg)
    print(f"User : {user_msg}")
    print(f"Bot  : {bot_reply}")
    print()

## 5. Interactive Chat (Notebook Cell)

The cell below runs a simple loop so you can type messages directly. Type `quit` to exit.

In [ ]:
def interactive_chat():
    """Run an interactive chat session in the notebook."""
    chat_bot = Chatbot()
    print("=" * 50)
    print("BlenderBot Chatbot — type 'quit' to exit, 'reset' to clear history")
    print("=" * 50)
    while True:
        try:
            user_input = input("You: ").strip()
        except (EOFError, KeyboardInterrupt):
            print("\n[Session ended]")
            break
        if not user_input:
            continue
        if user_input.lower() == "quit":
            print("Goodbye!")
            break
        if user_input.lower() == "reset":
            chat_bot.reset()
            continue
        reply = chat_bot.chat(user_input)
        print(f"Bot: {reply}\n")


# Uncomment to run interactively:
# interactive_chat()

## 6. Analysing Conversation History

Let's inspect the token counts as the history grows, to understand the context-window constraint.

In [ ]:
import matplotlib.pyplot as plt

# Rebuild conversation and track token counts
bot2 = Chatbot(max_history_turns=8)
token_counts = []

test_messages = [
    "Hello! Tell me something interesting.",
    "That's fascinating. Can you elaborate?",
    "How does that relate to everyday life?",
    "What would you say to someone who disagrees?",
    "Fair point. Is there anything else I should know?",
]

for msg in test_messages:
    _ = bot2.chat(msg)
    # Count tokens in the full history string fed to the model
    hist_str = "  ".join(bot2.history)
    n_tokens = len(tokenizer.encode(hist_str))
    token_counts.append(n_tokens)

plt.figure(figsize=(8, 4))
plt.plot(range(1, len(token_counts) + 1), token_counts, marker="o", color="coral")
plt.axhline(y=128, linestyle="--", color="gray", label="max_length=128 (truncation)")
plt.xlabel("Turn number")
plt.ylabel("Total tokens in history")
plt.title("Token count of conversation history over time")
plt.legend()
plt.tight_layout()
plt.show()

## 7. Extension Ideas

1. **Domain adaptation** – fine-tune BlenderBot on a customer-support dataset (e.g., Ubuntu Dialogue Corpus) to create a specialised support agent.
2. **Persona conditioning** – prepend a persona description (e.g., "You are a helpful cooking assistant.") to the context.
3. **Retrieval-augmented chat** – retrieve relevant documents and inject them into the prompt for knowledge-grounded responses.
4. **Gradio UI** – wrap `get_reply` with a Gradio `ChatInterface` for a polished web demo.

## 8. Summary

In this notebook we:
- Loaded BlenderBot and understood its dialogue format
- Built a stateful `Chatbot` class that maintains conversation history
- Observed how the context window grows with each turn
- Discussed extension ideas for domain adaptation and UI integration

**Next**: `03_translation.ipynb` — translating text between languages with T5.